In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from datetime import datetime, timedelta

In [ ]:
from aquacrop.utils import prepare_weather, get_filepath

In [ ]:
import rasterio as rio
import geopandas as gpd

In [ ]:
# repository root (notebooks live in <repo>/notebooks)
base_loc = os.path.abspath(os.path.join(os.getcwd(), '..'))
weather_loc=os.path.join(base_loc,'data','weather_aquacrop')
output_loc=os.path.join(base_loc,'output')

In [ ]:
def get_weather(dist):
    filepath=get_filepath(os.path.join(weather_loc,dist+'.txt'))
    weather_data = prepare_weather(filepath)
    return weather_data

In [ ]:
def season_rainfall(ptd,hd,dist):
    weather_data=get_weather(dist)
    weather_data=weather_data[['Date','Precipitation', 'MinTemp', 'MaxTemp','ReferenceET']]
    weather_data['Date']=pd.to_datetime(weather_data['Date'].astype(str),format='%Y-%m-%d')
    sd=datetime.strptime(ptd,'%m-%d').replace(year=hd.year)
    if sd>hd:
        sd=datetime.strptime(ptd,'%b %d').replace(year=hd.year-1)
    rf=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['Precipitation'].sum(),2))
    mi=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['MinTemp'].mean(),2))
    mx=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['MaxTemp'].mean(),2))
    et=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['ReferenceET'].mean(),2))
    return rf

In [ ]:
def season_mint(ptd,hd,dist):
    weather_data=get_weather(dist)
    weather_data=weather_data[['Date','Precipitation', 'MinTemp', 'MaxTemp','ReferenceET']]
    weather_data['Date']=pd.to_datetime(weather_data['Date'].astype(str),format='%Y-%m-%d')
    sd=datetime.strptime(ptd,'%m-%d').replace(year=hd.year)
    if sd>hd:
        sd=datetime.strptime(ptd,'%b %d').replace(year=hd.year-1)
    rf=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['Precipitation'].sum(),2))
    mi=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['MinTemp'].mean(),2))
    mx=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['MaxTemp'].mean(),2))
    et=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['ReferenceET'].mean(),2))
    return mi

In [ ]:
def season_maxt(ptd,hd,dist):
    weather_data=get_weather(dist)
    weather_data=weather_data[['Date','Precipitation', 'MinTemp', 'MaxTemp','ReferenceET']]
    weather_data['Date']=pd.to_datetime(weather_data['Date'].astype(str),format='%Y-%m-%d')
    sd=datetime.strptime(ptd,'%m-%d').replace(year=hd.year)
    if sd>hd:
        sd=datetime.strptime(ptd,'%b %d').replace(year=hd.year-1)
    rf=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['Precipitation'].sum(),2))
    mi=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['MinTemp'].mean(),2))
    mx=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['MaxTemp'].mean(),2))
    et=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['ReferenceET'].mean(),2))
    return mx

In [ ]:
def season_et(ptd,hd,dist):
    weather_data=get_weather(dist)
    weather_data=weather_data[['Date','Precipitation', 'MinTemp', 'MaxTemp','ReferenceET']]
    weather_data['Date']=pd.to_datetime(weather_data['Date'].astype(str),format='%Y-%m-%d')
    sd=datetime.strptime(ptd,'%m-%d').replace(year=hd.year)
    if sd>hd:
        sd=datetime.strptime(ptd,'%b %d').replace(year=hd.year-1)
    rf=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['Precipitation'].sum(),2))
    mi=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['MinTemp'].mean(),2))
    mx=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['MaxTemp'].mean(),2))
    et=(round(weather_data[(weather_data['Date']>=sd) & (weather_data['Date']<=hd)]['ReferenceET'].mean(),2))
    return et

In [ ]:
# Rajasthan district boundaries (subset of the all-India district shapefile)
india_shape = os.path.join(base_loc, 'data', 'location data', 'Rajasthan.shp')
data=gpd.read_file(india_shape)
raj=data[data['STATE_NAME']=='Rajasthan']
raj=raj.drop(columns=['Crop','ID'], errors='ignore')

In [ ]:
districts=list(raj['DISTRICT'].unique())

In [ ]:
final=pd.DataFrame()
for d in np.arange(1,8):
    req=pd.DataFrame()
    d=int((d-1)*15)
    for dist in districts:
        init_date=datetime.strptime('01-06','%d-%m')
        date=init_date+timedelta(days=d)
        planting_date=(datetime.strftime(date,'%B %d'))
        plt_dt=datetime.strftime(date,'%m-%d')
        fn=os.path.join(base_loc,'output','Final_stats_'+plt_dt+'-'+dist+'.xlsx')
        df=pd.read_excel(fn)
        df['Planting Date']=plt_dt
        df['District']=dist
        df['Rainfall']=df['Harvest Date (YYYY/MM/DD)'].apply(lambda x:season_rainfall(plt_dt,x,dist))
        df['Mint']=df['Harvest Date (YYYY/MM/DD)'].apply(lambda x:season_mint(plt_dt,x,dist))
        df['Maxt']=df['Harvest Date (YYYY/MM/DD)'].apply(lambda x:season_maxt(plt_dt,x,dist))
        df['ET']=df['Harvest Date (YYYY/MM/DD)'].apply(lambda x:season_et(plt_dt,x,dist))
        final=final.append(df)

In [ ]:
final.to_excel(os.path.join(base_loc,'output','compiled data.xlsx'))
